# Special features: SimOpt, inexactness, and SOL

**SIAM MDS 2026 minitutorial, Part IV.**

This notebook covers three special features:

1. **SimOpt:** tune a ridge weight through an implicitly defined state, in reduced and full space.
2. **Inexactness:** adapt the accuracy of state solves to ROL's requested tolerance.
3. **Stochastic Optimization Library (SOL):** fit a model against adverse distributions in a KL ball.

In [ ]:
%pip install -q --only-binary=:all: "rol-python==2026.11.9.dev1945"

In [ ]:
# ROL prints from C++. Colab needs wurlitzer to show that output in the notebook;
# other Jupyter kernels (e.g. Binder) show it already, and wurlitzer would hide it there.
import sys
if 'google.colab' in sys.modules:
    %pip install -q wurlitzer
    %load_ext wurlitzer

In [ ]:
import sys
from importlib.metadata import version

import numpy as np
import matplotlib.pyplot as plt
from pyrol import Objective, Problem, Solver, Bounds, getCout
from pyrol.pyrol.ROL import UpdateType
from pyrol.pyrol.Teuchos import ParameterList
from pyrol.unsupported import (BoundConstraint, BoundConstraint_SimOpt, Constraint_SimOpt,
                               Objective_SimOpt, Reduced_Objective_SimOpt, Vector_SimOpt)

print('Python', sys.version.split()[0], '| rol-python', version('rol-python'))

### Setup

We use PyROL's `NumPyVector` to wrap NumPy arrays. See the Quick Start notebook for an introduction to ROL vectors.

In [ ]:
from pyrol.vectors import NumPyVector

## 1. SimOpt

We generate 30 noisy observations of a smooth signal and define 24 fixed kernel functions $K(s,t)=\min(s,t)-st$, centred at $\tau_j=j/25$, $j=1,\ldots,24$. The dictionary matrix has entries $\Phi_{kj}=K(t_k,\tau_j)$. We fit their coefficients by ridge regression. We use $u$ for the fitted coefficients and $z$ for the regularization weight. We fit $u$ on the training data, then choose $z$ using validation data. This gives a bilevel optimization problem:

$$\min_{u,z} J(u,z)=\tfrac12\lVert\Phi'u-y'\rVert^2,\quad z\ge10^{-4},\qquad u=\arg\min_v\{\tfrac12\lVert\Phi v-y\rVert^2+\tfrac z2\lVert v\rVert^2\}.$$

The optimality condition for training gives us the state equation $c(u,z)=(\Phi^\top\Phi+zI)u-\Phi^\top y=0$. Solving for the state gives $S(z)=(\Phi^\top\Phi+zI)^{-1}\Phi^\top y$, and substituting this into the validation loss gives the reduced objective $J_{\mathrm{red}}(z)=J(S(z),z)$. We choose $z$ using 100 separate validation observations. $\Phi$ has rank 22; the positive lower bound on $z$ makes the training problem strictly convex, so it has a unique solution.

In [ ]:
from scipy.linalg import cho_factor, cho_solve


rng = np.random.default_rng(20261012)
t_obs = np.sort(rng.uniform(0.03, 0.97, 30))
y_obs = 0.8*np.sin(2*np.pi*t_obs) + 0.27*np.sin(5*np.pi*t_obs) + rng.normal(scale=0.12, size=30)


centers = np.arange(1, 25)/25
T, C = np.meshgrid(t_obs, centers, indexing='ij')
Phi = np.minimum(T, C) - T*C          # Phi[i, j] = K(t_i, c_j)


rng = np.random.default_rng(20261014)
t_val = np.sort(rng.uniform(0.03, 0.97, 100))
y_val = 0.8*np.sin(2*np.pi*t_val) + 0.27*np.sin(5*np.pi*t_val) + rng.normal(scale=0.12, size=100)
Phi_val = np.minimum(t_val[:, None], centers) - t_val[:, None]*centers   # Phi'[j, k] = K(t'_j, tau_k)

M, b = Phi.T @ Phi, Phi.T @ y_obs          # the normal equations of the training problem
N, EPS, Z0 = len(centers), 1e-4, 0.08     # Z0: the weight of Part II's example
print('rank of Phi:', np.linalg.matrix_rank(Phi))

In [ ]:
# Training observations y at the observation times t.
fig, ax = plt.subplots(figsize=(7, 3.5))
t_true = np.linspace(0, 1, 500)
y_true = 0.8*np.sin(2*np.pi*t_true) + 0.27*np.sin(5*np.pi*t_true)
ax.plot(t_true, y_true, '-', label='True signal (noise-free)')
ax.plot(t_obs, y_obs, 'o', label='Training observations')
ax.set(xlabel='t', ylabel='y', title='Training data')
ax.legend()
fig.tight_layout()
plt.show()

### 1.1. The state equation and partial derivatives

`StateEquation` implements $c(u,z)=0$, the state solve, and the partial derivatives through `Constraint_SimOpt`. State and adjoint solves reuse a factorization of $\Phi^\top\Phi+zI$, managed by ROL's update calls.

`Reduced_Objective_SimOpt` combines this constraint with `ValidationLoss` to form an objective in $z$ alone. It computes the reduced gradient using state and adjoint solves; here, Hessian-vector products use finite differences of that gradient. The lower bound on $z$ makes this a Type B problem.

In [ ]:
class Factorization:
    """The Cholesky factorization of a positive definite matrix A, computed at the first solve."""
    def __init__(self, A):
        self.A, self.chol = A, None

    def solve(self, v):
        if self.chol is None:
            self.chol = cho_factor(self.A)
        return cho_solve(self.chol, v)


class StateEquation(Constraint_SimOpt):
    """c(u, z) = (Phi^T Phi + z I) u - Phi^T y, the training problem's optimality condition."""
    def value(self, c, u, z, tol):
        c.array[:] = M @ u.array + z.array[0]*u.array - b

    def applyJacobian_1(self, jv, v, u, z, tol):
        jv.array[:] = M @ v.array + z.array[0]*v.array

    def solve(self, c, u, z, tol):
        u.array[:] = self.factor.solve(b)
        self.value(c, u, z, tol)

    def update_2(self, z, type, iter):
        if type == UpdateType.Accept:    # the trial is the new iterate
            self.iterate = self.factor
        elif type == UpdateType.Revert:  # back to the iterate
            self.factor = self.iterate
        else:                            # a new control
            self.factor = Factorization(M + z.array[0]*np.eye(N))
            if type == UpdateType.Initial:
                self.iterate = self.factor

    def applyAdjointJacobian_1(self, ajv, v, u, z, tol):     # symmetric
        ajv.array[:] = M @ v.array + z.array[0]*v.array

    def applyInverseJacobian_1(self, ijv, v, u, z, tol):
        ijv.array[:] = self.factor.solve(v.array)

    def applyInverseAdjointJacobian_1(self, iajv, v, u, z, tol):
        iajv.array[:] = self.factor.solve(v.array)

    def applyJacobian_2(self, jv, v, u, z, tol):             # (dc/dz) v = v u
        jv.array[:] = v.array[0]*u.array

    def applyAdjointJacobian_2(self, ajv, v, u, z, tol):
        ajv.array[:] = u.array @ v.array


class ValidationLoss(Objective_SimOpt):
    """J(u, z) = |Phi' u - y'|^2 / 2, the squared error on the validation observations."""
    def value(self, u, z, tol):
        r = Phi_val @ u.array - y_val
        return 0.5*(r @ r)

    def gradient_1(self, g, u, z, tol):
        g.array[:] = Phi_val.T @ (Phi_val @ u.array - y_val)

    def gradient_2(self, g, u, z, tol):                      # J does not depend on z
        g.array[:] = 0.0


def parameters():
    params = ParameterList()
    params['General'] = ParameterList()
    params['General']['Output Level'] = 1
    params['Step'] = ParameterList()
    params['Step']['Type'] = 'Trust Region'
    params['Status Test'] = ParameterList()
    params['Status Test']['Gradient Tolerance'] = 1e-8
    params['Status Test']['Iteration Limit'] = 50
    return params


def lower_bound():
    return Bounds(NumPyVector(np.array([EPS])), True)        # z >= eps


def lower_bound_simopt():
    """z >= eps on the pair (u, z), with no bound on the state."""
    free = BoundConstraint()
    free.deactivate()
    return BoundConstraint_SimOpt(free, lower_bound())


def solve(z0=Z0):
    z = NumPyVector(np.array([z0]))
    u = NumPyVector(np.zeros(N))                   # the state
    psi = NumPyVector(np.zeros(N))                 # the adjoint
    reduced = Reduced_Objective_SimOpt(ValidationLoss(), StateEquation(),
                                       u, z, psi, True, True)
    problem = Problem(reduced, z)
    problem.addBoundConstraint(lower_bound())
    stream = getCout()
    Solver(problem, parameters()).solve(stream)
    return z.array[0]


def full_space_parameters():
    params = parameters()
    params['Step']['Type'] = 'Augmented Lagrangian'
    params['Step']['Augmented Lagrangian'] = ParameterList()
    params['Step']['Augmented Lagrangian']['Use Default Problem Scaling'] = False
    params['Status Test']['Gradient Tolerance'] = 1e-6
    params['Status Test']['Constraint Tolerance'] = 1e-8
    params['Status Test']['Iteration Limit'] = 100
    return params


def solve_full_space(z0=Z0):
    z = NumPyVector(np.array([z0]))
    u = NumPyVector(Factorization(M + z0*np.eye(N)).solve(b))   # feasible start
    x = Vector_SimOpt(u, z)                        # the pair (u, z)
    problem = Problem(ValidationLoss(), x)
    lam = NumPyVector(np.zeros(N))                 # its multiplier
    problem.addConstraint('state', StateEquation(), lam)
    problem.addBoundConstraint(lower_bound_simopt())
    stream = getCout()
    Solver(problem, full_space_parameters()).solve(stream)
    return z.array[0], lam.array.copy()


def reduced_value(z):
    """The reduced objective J(S(z), z) at the number z, by a direct solve (a reference)."""
    u = np.linalg.solve(M + z*np.eye(N), b)
    r = Phi_val @ u - y_val
    return 0.5*(r @ r)


def reduced_derivative(z):
    """-psi^T u with (Phi^T Phi + z I) psi = Phi'^T (Phi' u - y'): the adjoint formula (a reference)."""
    A = M + z*np.eye(N)
    u = np.linalg.solve(A, b)
    psi = np.linalg.solve(A, Phi_val.T @ (Phi_val @ u - y_val))
    return -psi @ u

### 1.2. Reduced-space solve and derivative check

Let's use ROL's finite-difference checker to check the reduced gradient computed with the adjoint equation.

The checker compares the directional derivative from our gradient (`grad'*dir`) with a finite-difference approximation (`FD approx`). Their difference appears in the `abs error` column. As the step size gets smaller, we expect this error to decrease, roughly by a factor of ten when the step shrinks by ten, since the default check uses a first-order difference. For very small steps, cancellation and roundoff take over, so the error may level off or rise again. Look for a clear decrease to a small error before that happens; we don't expect the error to keep shrinking all the way down the table.

In [ ]:
z0 = NumPyVector(np.array([Z0]))
direction = NumPyVector(np.array([0.01]))
reduced = Reduced_Objective_SimOpt(ValidationLoss(), StateEquation(), NumPyVector(np.zeros(N)),
                                   z0, NumPyVector(np.zeros(N)), True, True)
stream = getCout()
reduced.checkGradient(z0, direction, True, stream)
stream.flush();

Now let's solve the reduced problem for the ridge weight $z$. The plot shows how the validation error changes with $z$, with the solution marked by a dot.

In [ ]:
z_star = solve()
print('z =', z_star, ' validation error', reduced_value(Z0), '->', reduced_value(z_star))

ridge_weights = np.geomspace(EPS, 0.3, 200)
plt.semilogx(ridge_weights, [reduced_value(g) for g in ridge_weights])
plt.plot(z_star, reduced_value(z_star), 'o')
plt.xlabel('z (ridge weight)'); plt.ylabel('validation error'); plt.show()

### 1.3. Full-space solve

We can use the same two classes to solve for $u$ and $z$ together. The state equation and the bound on $z$ give us a Type G problem. `Vector_SimOpt` holds the pair $(u,z)$, and `solve_full_space` uses ROL's augmented Lagrangian method.

**Before running:** do you expect more or fewer iterations than in reduced space?

In [ ]:
z_full, lam = solve_full_space()
print('full space: z =', z_full, ' validation error', reduced_value(z_full))
assert abs(z_full-z_star) < 1e-7
assert abs(reduced_derivative(z_star)) < 1e-5

Both formulations should find $z\approx7.563\times10^{-4}$ with validation loss about 0.96545. In this example the full-space augmented Lagrangian needs more outer iterations than the reduced-space trust region. The two algorithms do different amounts of work per iteration, so we can't compare their cost just by counting iterations.

### 1.4. Questions

Which methods supply state and adjoint solves for the reduced gradient? Why do full and reduced space give the same validation loss? Which partial derivatives use suffix `_1`, and which use `_2`?

## 2. Inexactness

Objective and gradient evaluations can be expensive. ROL's inexact algorithms let us use cheaper approximations early on and ask for more accuracy as needed. For an inexact trust-region method, we need to control the error in the **objective reduction** used to accept a step. Gradient errors also need to stay small relative to the gradient norm and trust-region radius. In this example, we approximate only the objective values; the gradients use direct state and adjoint solves.

### 2.1. Bounding the value error using the residual

Let us return to the same state equation from the previous section. For an approximate state $\tilde u$, let $A=\Phi^\top\Phi+zI$ and $r=b-A\tilde u$, where $b=\Phi^\top y$. Since $A\succeq zI$, $\lVert u-\tilde u\rVert\le\delta=\lVert r\rVert/z$. With $\tilde e=\Phi'\tilde u-y'$ and $\beta=\lVert\Phi'\rVert_F$,

$$|J(u,z)-J(\tilde u,z)|\le\lVert\tilde e\rVert\,\beta\delta+\tfrac12(\beta\delta)^2.$$

The `value` method starts from the previous approximate state and uses SciPy's `cg` to improve it. After each CG solve, the outer loop recomputes the residual and checks the value-error bound. It returns the approximate objective value once the bound is at most `tol`. `RecordedLoss` keeps track of the tolerances and error bounds without changing how we solve for the state.

In [ ]:
from scipy.sparse.linalg import cg
BETA = np.linalg.norm(Phi_val, ord="fro")


class ScipyInexactLoss(Objective):
    """Warm-started CG states with a bound on the validation-loss error."""
    def __init__(self):
        super().__init__()
        self.u = np.zeros(N)

    def value(self, z, tol):
        z = z.array[0]
        A = M + z*np.eye(len(b))
        for _ in range(50):
            r = b - A @ self.u
            e = Phi_val @ self.u - y_val
            t = BETA * np.linalg.norm(r) / z
            if np.linalg.norm(e)*t + 0.5*t**2 <= tol:
                return 0.5 * (e @ e)
            self.u, info = cg(A, b, x0=self.u, rtol=0,
                             atol=0.1*np.linalg.norm(r))
            if info != 0: raise RuntimeError("CG failed")
        raise RuntimeError("Accuracy not reached")

    def gradient(self, g, z, tol):
        g.array[:] = reduced_derivative(z.array[0])


class RecordedLoss(ScipyInexactLoss):
    """Record requested/effective tolerances and residual-based error bounds."""
    def __init__(self, fixed_tol=None):
        super().__init__()
        self.fixed_tol = fixed_tol
        self.calls = []

    def value(self, z, tol):
        requested = tol if self.fixed_tol is None else self.fixed_tol
        result = super().value(z, requested)
        control = z.array[0]
        residual = b - (M + control*np.eye(len(b))) @ self.u
        t = BETA*np.linalg.norm(residual)/control
        e = Phi_val @ self.u - y_val
        bound = np.linalg.norm(e)*t + 0.5*t**2
        self.calls.append((requested, bound))
        return result


def parameters(inexact=True):
    params = ParameterList()
    params['General'] = ParameterList()
    params['General']['Output Level'] = 1
    params['General']['Inexact Objective Function'] = inexact
    params['Step'] = ParameterList()
    params['Step']['Type'] = 'Trust Region'
    params['Status Test'] = ParameterList()
    params['Status Test']['Gradient Tolerance'] = 5e-2
    params['Status Test']['Iteration Limit'] = 50
    return params


def solve(objective, inexact=True):
    z = NumPyVector(np.array([Z0]))
    problem = Problem(objective, z)
    problem.addBoundConstraint(lower_bound())
    stream = getCout()
    Solver(problem, parameters(inexact)).solve(stream)
    return z.array[0]

### 2.2. Let ROL choose the accuracy

Turning on `General / Inexact Objective Function` makes ROL select the tolerance passed to `value(z, tol)`. The first tolerance is very large, so the initial state needs little or no improvement. Later, ROL may reevaluate both the current and trial points using the same tighter tolerance. The stopping tolerance for the outer gradient is $0.05$.

In [ ]:
adaptive = RecordedLoss()
z_adaptive = solve(adaptive, inexact=True)
calls = np.array(adaptive.calls)
tols, error_bounds = calls[:, 0], calls[:, 1]
print('z:', z_adaptive, ' value calls:', len(calls))
print('distance to the exact solution:', abs(z_adaptive-z_star))
assert np.all(error_bounds <= tols)
assert abs(z_adaptive-z_star)/z_star < 2e-4
plt.semilogy(np.where(tols < 1e300, tols, np.nan), 'o-', label='Requested by ROL')
plt.semilogy(error_bounds, 's', label='Error bound')
plt.xlabel('Value call'); plt.legend(); plt.show()

### 2.3. Questions

Why do requested tolerances tighten near convergence? Why do we need more than the state residual to bound the validation-loss error? Try tightening the outer gradient tolerance. Roundoff in the residual gets amplified by $1/z$, so eventually we hit the limits of double precision. If the error bound stays above the requested tolerance, the routine raises an error.

## 3. The Stochastic Optimization Library (SOL)

SOL handles objectives with random inputs. Here we'll use distributionally robust optimization (DRO) to account for uncertainty in the probability distribution itself.

### 3.1. Regression under distributional uncertainty

Let's try a new regression dataset: 48 fixed nonlinear features, 600 training observations, and 2400 held-out observations. The features are nonlinear, but the model is still linear in the coefficients $u$ that we fit. About 18% of the observations form a harder subgroup with a different target relationship. We use the subgroup labels only to evaluate the fits.

Let $\ell_i(u)=\tfrac12(h_i^\top u-y_i)^2$, with nominal probabilities $p_i=1/N$. The mean fit minimizes the average loss plus ridge regularization. For DRO, we allow distributions in an ambiguity set and minimize the worst-case expected loss. We keep the observed inputs and responses fixed and let their probabilities change:

$$Q_\rho=\{q\ge0:\ \sum_iq_i=1,\ \mathrm{KL}(q\Vert p)=\sum_iq_i\log(q_i/p_i)\le\rho\},$$
$$\min_u\max_{q\in Q_\rho}\sum_iq_i\ell_i(u)+\frac z2\lVert u\rVert^2.$$

We set $\rho=0.1$, and $z=0.002$ for this example.

In [ ]:
rng = np.random.default_rng(20261012)
N_TRAIN, N_TEST, M, D = 600, 2400, 48, 48

q_left, _ = np.linalg.qr(rng.normal(size=(D, M)))
q_right, _ = np.linalg.qr(rng.normal(size=(M, M)))
A = q_left @ np.diag(np.geomspace(1.0, 0.085, M)) @ q_right.T


def feature_map(X):
    return np.tanh((X @ A) * np.sqrt(M) / 2.0)


X_train = rng.normal(size=(N_TRAIN, D))
H_train = feature_map(X_train)
eigenvalues, eigenvectors = np.linalg.eigh(H_train.T @ H_train / N_TRAIN)
w_base = rng.normal(size=M) / np.sqrt(M)
w_shift = eigenvectors[:, -1] * (3.0 / np.sqrt(eigenvalues[-1]))


def generate_targets(H, is_rare):
    return H @ w_base + is_rare * (H @ w_shift) + rng.normal(0, 0.18, len(H))


rare_train = (rng.random(N_TRAIN) < 0.18)
y_train = generate_targets(H_train, rare_train)
X_test = rng.normal(size=(N_TEST, D))
H_test = feature_map(X_test)
rare_test = (rng.random(N_TEST) < 0.18)
y_test = generate_targets(H_test, rare_test)

### 3.2. The loss of one observation and the nominal distribution

Each sample point is a complete observation $\xi=(h_i,y_i)$: a row of the feature matrix followed by its response. `observations(H, y, folder)` writes these rows to `points.txt` and nominal probabilities $1/N$ to `weights.txt`. The sample dimension is the number of features plus one (49 here).

`SampleLoss(z)` takes the ridge weight $z$ as an input. Its `setParameter` method receives each observation's feature vector and response. `value`, `gradient`, and `hessVec` use that observation directly. SOL handles the worst-case reweighting. Including the ridge penalty in each sample loss gives the usual ridge penalty after averaging.

In [ ]:
import tempfile
from pathlib import Path
from pyrol import getParametersFromYamlFile
from pyrol.unsupported import BatchManager, StochasticProblem, UserInputGenerator

# Embed the complete parameter file: no repository files are read.
_parameter_dir = tempfile.TemporaryDirectory()
PARAMS = Path(_parameter_dir.name) / 'sol_kl.yaml'
_ = PARAMS.write_text('Inputs:\n  General:\n    Output Level: 1\n  Step:\n    Type: Trust Region\n    Trust Region:\n      Subproblem Solver: Truncated CG\n  Status Test:\n    Gradient Tolerance: 1.0e-8\n    Iteration Limit: 200\n  SOL:\n    Objective:\n      Type: Risk Averse\n      Risk Measure:\n        Name: KL Divergence\n        KL Divergence:\n          Threshold: 0.1\n')

In [ ]:
class SampleLoss(Objective):
    def __init__(self, z):
        super().__init__()
        self.z = z

    def setParameter(self, sample):
        sample = np.array(sample)
        self.h, self.y = sample[:-1], sample[-1]

    def value(self, u, tol):
        u = u.array
        r = self.h @ u - self.y
        return 0.5 * (r*r + self.z * (u @ u))

    def gradient(self, g, u, tol):
        r = self.h @ u.array - self.y
        g.array[:] = r*self.h + self.z*u.array

    def hessVec(self, hv, v, u, tol):
        hv.array[:] = (self.h @ v.array)*self.h + self.z*v.array


def observations(H, y, folder):
    """Read the empirical distribution from full data rows and uniform weights."""
    H, y = np.asarray(H), np.asarray(y)
    if H.ndim != 2 or y.ndim != 1 or H.shape[0] != len(y) or not len(y):
        raise ValueError('H must have one row per response in nonempty y')
    points, weights = Path(folder) / 'points.txt', Path(folder) / 'weights.txt'
    N, d = H.shape[0], H.shape[1] + 1
    np.savetxt(points, np.column_stack((H, y)))
    np.savetxt(weights, np.full(N, 1/N))
    return UserInputGenerator(str(points), str(weights), N, d, BatchManager())

In [ ]:
def parameters(radius=0.1, output_level=1):
    """Select the native SOL KL risk measure; radius is measured in natural-log units."""
    if not np.isfinite(radius) or radius <= 0:
        raise ValueError('radius must be finite and strictly positive')
    params = getParametersFromYamlFile(str(PARAMS))
    params['SOL']['Objective']['Risk Measure']['KL Divergence']['Threshold'] = float(radius)
    params['General']['Output Level'] = int(output_level)
    return params


def tilted_weights(loss, temperature):
    """Return adverse probabilities and log-mean-exp with a stable exponential shift.

    The second return value is log(mean(exp((loss-max(loss))/temperature))).
    """
    loss = np.asarray(loss, dtype=float)
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError('temperature must be finite and strictly positive')
    z = (loss - loss.max()) / temperature
    weights = np.exp(z)
    return weights / weights.sum(), float(np.log(weights.mean()))

### 3.3. KL-DRO in PyROL

Now let's load the KL-DRO settings and solve. The parameter list selects `Risk Measure / Name: KL Divergence` and `KL Divergence / Threshold: 0.1`. `StochasticProblem` brings the sample loss and sampler together. We call `makeObjectiveStochastic` to set up the objective, then `Solver` to compute the fit.

For comparison, the mean fit uses the same data and regularization with `SOL / Objective / Type` set to `Risk Neutral`. Both fits use $z=0.002$. The YAML settings are included above, so we don't need a separate file.

In [ ]:
rho, z = 0.1, 0.002
H, y = H_train, y_train
with tempfile.TemporaryDirectory() as folder:
    sampler = observations(H, y, folder)

    # Mean fit: the same observations and ridge weight, with nominal probabilities.
    mean_params = parameters(rho)
    mean_params['SOL']['Objective']['Type'] = 'Risk Neutral'
    u_mean = NumPyVector(np.zeros(H.shape[1]))
    mean_problem = StochasticProblem(SampleLoss(z), u_mean)
    mean_problem.makeObjectiveStochastic(mean_params, sampler)
    Solver(mean_problem, mean_params).solve(getCout())
    w_mean = u_mean.array.copy()

    # KL-DRO: minimize the worst-case expected loss over the KL ambiguity set.
    u = NumPyVector(w_mean.copy())
    params = getParametersFromYamlFile(str(PARAMS))
    params['SOL']['Objective']['Risk Measure']['KL Divergence']['Threshold'] = rho
    problem = StochasticProblem(SampleLoss(z), u)
    problem.makeObjectiveStochastic(params, sampler)
    Solver(problem, params).solve(getCout())
    w_dro = u.array.copy()

### 3.4. KL-DRO regression results

Compare the nominal mean loss and the KL worst-case expected loss for both fits. Nominal loss uses uniform probabilities. For each model, the worst-case loss uses the most damaging reweighting allowed by the same KL ball, with radius $\rho=0.1$.

We leave out the ridge penalty when reporting these training losses. Expect approximately **0.60/1.59** for the mean fit and **0.72/1.31** for KL-DRO. KL-DRO trades higher nominal loss for lower worst-case loss.

**Before running:** if the worst-case expected loss goes down, does the nominal mean loss have to go down too?

In [ ]:
from scipy.optimize import brentq  # scalar root finder, not a model optimizer

def worst_case_weights(loss, radius):
    """Adverse probabilities for this example's small KL radii (0 through 0.2)."""
    if radius == 0:
        return np.full(len(loss), 1/len(loss))
    def residual(log_eta):
        temperature = np.exp(log_eta)
        weights, log_mean = tilted_weights(loss, temperature)
        return weights @ ((loss-loss.max())/temperature) - log_mean - radius
    temperature = np.exp(brentq(residual, -20, 30))
    return tilted_weights(loss, temperature)[0]

train_losses, adverse = {}, {}
print(f'{"training loss":16s} {"nominal mean":>14s} {"KL worst case":>14s}')
for name, w in [('Mean fit', w_mean), ('KL-DRO', w_dro)]:
    loss = 0.5*(H_train @ w - y_train)**2
    weights = worst_case_weights(loss, rho)
    train_losses[name], adverse[name] = loss, weights
    print(f'{name:16s} {loss.mean():14.4f} {weights @ loss:14.4f}')

### 3.5. Questions and experiments

1. Change the KL radius to 0.05 or 0.2 and refit. How do the nominal mean loss and KL worst-case expected loss change? Compare both models within the same ambiguity set.
2. Why does enlarging the ambiguity set increase the worst-case expected loss for a fixed model?
3. Does lower worst-case expected training loss guarantee lower expected loss on new observations?